# Heart Disease Risk Prediction — Logistic Regression
**Project:** AI/ML Course — Cleveland UCI Heart Disease Dataset
**Deadline:** Aug 11

Pipeline: Load → Clean → Feature Selection → Train (Logistic Regression + baselines) →
Statistical Validation (CV, LOOCV, Wald test, LR test, odds ratios) → Save model for live demo.

⚠️ **গুরুত্বপূর্ণ:** যেকোনো সমস্যা হলে সবার আগে `Runtime → Restart session and run all` করো —
সেলগুলো এলোমেলো ক্রমে বা একাধিকবার রান করলে ডেটা নষ্ট হয়ে যেতে পারে।

In [ ]:
!pip install statsmodels -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, LeaveOneOut, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score, precision_score,
                              recall_score, confusion_matrix, classification_report,
                              roc_curve, ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve

import statsmodels.api as sm

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded successfully.")

## ১. Data Load
নিজের `processed_cleveland.csv` আপলোড করো। এই সেলটা নিজে থেকেই বুঝে নেবে ফাইলে header আছে কিনা,
আর "?" চিহ্নকে মিসিং ভ্যালু হিসেবে ধরবে।

In [ ]:
from google.colab import files
print("Please select processed_cleveland.csv to upload:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

expected_cols = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
                  "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]

# First try assuming the file already has a header row
df = pd.read_csv(filename, na_values="?")
df.columns = [str(c).strip().lower() for c in df.columns]

# If columns don't look right, re-read assuming there was no header
if not set(["age", "sex", "cp"]).issubset(set(df.columns)):
    df = pd.read_csv(filename, header=None, names=expected_cols, na_values="?")
    print("No header detected in file -> applied standard UCI column names.")
else:
    print("Header detected in file -> using existing column names.")

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
df.head()

In [ ]:
# Force every column to numeric, in case of stray characters/spaces.
# Values that cannot be converted become NaN instead of crashing later steps.
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Missing values per column:\n", df.isnull().sum())

## ২. Data Cleaning
- Target (0-4) কে binary (0 = no disease, 1 = disease) এ থ্রেশহোল্ড করা হচ্ছে
- Missing value (`ca`, `thal` কলামে সামান্য থাকে) হ্যান্ডেল করা হচ্ছে

এই সেলটা **idempotent** — মানে একাধিকবার রান করলেও সমস্যা হবে না, কারণ `target` কলাম আগে থেকে থাকলে
এটা সেটা আবার তৈরি করার চেষ্টা করবে না।

In [ ]:
if "target" not in df.columns:
    # Common names the raw disease-indicator column might have
    possible_names = ["num", "condition", "diagnosis", "class", "output", "heartdisease"]
    target_col = next((c for c in possible_names if c in df.columns), None)

    if target_col is None:
        target_col = df.columns[-1]
        print(f"Warning: no known target-column name found, falling back to last column: '{target_col}'")
    else:
        print(f"Target column found: '{target_col}'")

    print(f"Raw values in '{target_col}':\n", df[target_col].value_counts())

    df["target"] = (df[target_col] > 0).astype(int)
    df = df.drop(columns=[target_col])
else:
    print("'target' column already exists -- skipping re-creation.")

# Impute any remaining missing values with the column mode
for col in df.columns:
    if col != "target" and df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values (final check):\n", df.isnull().sum())
print("\nClass balance:\n", df["target"].value_counts())

assert df["target"].nunique() == 2, (
    "ERROR: target column has only one class. "
    "Restart the session and run all cells in order from the top."
)
print("\nSanity check passed: target has both classes.")

## ৩. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.countplot(x="target", data=df, ax=axes[0])
axes[0].set_title("Class Distribution")
sns.boxplot(x="target", y="thalach", data=df, ax=axes[1])
axes[1].set_title("Max Heart Rate vs Target")
plt.tight_layout()
plt.show()

## ৪. Preprocessing
- Categorical (`cp`, `restecg`, `slope`, `thal`) → one-hot encode
- Continuous ফিচার স্ট্যান্ডার্ডাইজ
- Train/Test split (stratified, 80/20)

In [ ]:
categorical_cols = [c for c in ["cp", "restecg", "slope", "thal"] if c in df.columns]
df_enc = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_enc.drop(columns=["target"])
y = df_enc["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

continuous_cols = [c for c in ["age","trestbps","chol","thalach","oldpeak","ca"] if c in X.columns]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("y_train class balance:\n", y_train.value_counts())
print("\ny_test class balance:\n", y_test.value_counts())

X_train_scaled.head()

## ৫. Feature Selection (RFE)

In [ ]:
assert y_train.nunique() == 2, (
    "ERROR: y_train has only one class. Restart the session and run all cells in order."
)

base_lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
rfe = RFE(base_lr, n_features_to_select=8)
rfe.fit(X_train_scaled, y_train)

selected_features = X_train_scaled.columns[rfe.support_].tolist()
print("Selected features:", selected_features)

X_train_sel = X_train_scaled[selected_features]
X_test_sel = X_test_scaled[selected_features]

## ৫.১ Genetic Algorithm দিয়ে Feature Selection (Extra Novelty)
RFE-এর পাশাপাশি একটা Genetic Algorithm দিয়েও feature subset খোঁজা হচ্ছে —
fitness হিসেবে stratified 5-fold cross-validated ROC-AUC ব্যবহার করা হচ্ছে।
⚠️ এটা রান হতে কয়েক মিনিট সময় নিতে পারে, ধৈর্য ধরে অপেক্ষা করো।

In [ ]:
import random

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

feature_pool = X_train_scaled.columns.tolist()
n_features = len(feature_pool)

POP_SIZE = 20
N_GENERATIONS = 30
MUTATION_RATE = 0.1
CX_RATE = 0.7

def create_individual():
    while True:
        ind = [random.randint(0, 1) for _ in range(n_features)]
        if sum(ind) >= 3:
            return ind

def fitness(individual):
    selected = [feature_pool[i] for i, bit in enumerate(individual) if bit == 1]
    if len(selected) < 3:
        return 0.0
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    scores = cross_val_score(model, X_train_scaled[selected], y_train,
                              cv=StratifiedKFold(5), scoring="roc_auc")
    penalty = 0.001 * len(selected)
    return scores.mean() - penalty

def tournament_select(pop, fitnesses, k=3):
    idxs = random.sample(range(len(pop)), k)
    best_idx = max(idxs, key=lambda i: fitnesses[i])
    return pop[best_idx][:]

def crossover(p1, p2):
    if random.random() < CX_RATE:
        point = random.randint(1, n_features - 1)
        return p1[:point] + p2[point:], p2[:point] + p1[point:]
    return p1[:], p2[:]

def mutate(ind):
    return [1 - bit if random.random() < MUTATION_RATE else bit for bit in ind]

population = [create_individual() for _ in range(POP_SIZE)]
best_individual = None
best_fitness = -1

for gen in range(N_GENERATIONS):
    fitnesses = [fitness(ind) for ind in population]

    gen_best_idx = np.argmax(fitnesses)
    if fitnesses[gen_best_idx] > best_fitness:
        best_fitness = fitnesses[gen_best_idx]
        best_individual = population[gen_best_idx][:]

    print(f"Generation {gen+1}/{N_GENERATIONS} - Best ROC-AUC so far: {best_fitness:.4f}")

    new_population = [population[gen_best_idx][:]]
    while len(new_population) < POP_SIZE:
        parent1 = tournament_select(population, fitnesses)
        parent2 = tournament_select(population, fitnesses)
        child1, child2 = crossover(parent1, parent2)
        new_population.append(mutate(child1))
        if len(new_population) < POP_SIZE:
            new_population.append(mutate(child2))

    population = new_population

ga_selected_features = [feature_pool[i] for i, bit in enumerate(best_individual) if bit == 1]
print("\nGA-selected features:", ga_selected_features)
print(f"GA best CV ROC-AUC: {best_fitness:.4f}")

In [ ]:
X_train_ga = X_train_scaled[ga_selected_features]
X_test_ga = X_test_scaled[ga_selected_features]

lr_ga = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_ga.fit(X_train_ga, y_train)

pred_ga = lr_ga.predict(X_test_ga)
prob_ga = lr_ga.predict_proba(X_test_ga)[:, 1]

lr_rfe_check = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_rfe_check.fit(X_train_sel, y_train)
pred_rfe_check = lr_rfe_check.predict(X_test_sel)
prob_rfe_check = lr_rfe_check.predict_proba(X_test_sel)[:, 1]

print("=== RFE-selected features ===")
print("Features:", selected_features)
print("Test Accuracy:", accuracy_score(y_test, pred_rfe_check))
print("Test ROC-AUC:", roc_auc_score(y_test, prob_rfe_check))

print("\n=== GA-selected features ===")
print("Features:", ga_selected_features)
print("Test Accuracy:", accuracy_score(y_test, pred_ga))
print("Test ROC-AUC:", roc_auc_score(y_test, prob_ga))

print("\nNote: if GA's ROC-AUC is higher, you can switch the rest of the pipeline")
print("to use ga_selected_features instead of selected_features (see markdown note below).")

**GA বেশি ভালো হলে যা করবে:** নিচের "৬. Hyperparameter Tuning (GridSearchCV)" সেলে,
`grid.fit(X_train_sel, y_train)` লাইনের ঠিক আগে এই লাইনটা যোগ করো:
```python
X_train_sel, X_test_sel = X_train_ga, X_test_ga
```
এটা করলে বাকি পুরো pipeline (tuning, CV, evaluation, saving) GA-selected feature দিয়ে চলবে।
RFE-ই ভালো থাকলে কিছু বদলানোর দরকার নেই।

## ৬. Hyperparameter Tuning (GridSearchCV)

In [ ]:
param_grid = {
    "C": [0.01, 0.05, 0.1, 0.5, 1, 5, 10],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"]
}

grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                     param_grid, cv=StratifiedKFold(5), scoring="roc_auc")
grid.fit(X_train_sel, y_train)

print("Best parameters:", grid.best_params_)
best_lr = grid.best_estimator_

## ৭. Cross-Validation দিয়ে Robustness যাচাই
Dataset ছোট (303 samples) বলে Stratified 5-Fold এর পাশাপাশি Leave-One-Out CV ও করা হচ্ছে।

In [ ]:
cv_scores = cross_val_score(best_lr, X_train_sel, y_train, cv=StratifiedKFold(5), scoring="accuracy")
print(f"Stratified 5-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# LOOCV (may take a minute)
loo = LeaveOneOut()
loo_scores = cross_val_score(best_lr, X_train_sel, y_train, cv=loo, scoring="accuracy")
print(f"LOOCV Accuracy: {loo_scores.mean():.4f}")

## ৮. Test Set Evaluation

In [ ]:
y_pred = best_lr.predict(X_test_sel)
y_prob = best_lr.predict_proba(X_test_sel)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot()
plt.title("Confusion Matrix")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_prob):.3f}")
plt.plot([0,1],[0,1],"--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

## ৯. Calibration Curve (predicted probability reliable কিনা)

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)
plt.plot(prob_pred, prob_true, marker="o", label="Model")
plt.plot([0,1],[0,1],"--", color="gray", label="Perfectly calibrated")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration Curve")
plt.legend()
plt.show()

## ১০. Statistical Significance — Wald Test, Odds Ratios, Likelihood Ratio Test
`statsmodels` দিয়ে p-value ও odds ratio (95% CI) বের করা হচ্ছে।

In [ ]:
X_sm = sm.add_constant(X_train_sel.astype(float))
logit_model = sm.Logit(y_train, X_sm).fit(disp=0)
print(logit_model.summary())

# Odds ratios + 95% CI
odds_ratios = pd.DataFrame({
    "OR": np.exp(logit_model.params),
    "CI_lower": np.exp(logit_model.conf_int()[0]),
    "CI_upper": np.exp(logit_model.conf_int()[1]),
    "p_value": logit_model.pvalues
})
print("\nOdds Ratios:\n", odds_ratios)

In [ ]:
# Likelihood Ratio Test: full model vs null model
null_model = sm.Logit(y_train, sm.add_constant(pd.Series(np.ones(len(y_train)), index=y_train.index, name="const"))).fit(disp=0)
lr_stat = 2 * (logit_model.llf - null_model.llf)
from scipy.stats import chi2
lr_pvalue = chi2.sf(lr_stat, df=len(selected_features))
print(f"Likelihood Ratio Test: LR-stat={lr_stat:.3f}, p-value={lr_pvalue:.6f}")

## ১১. Baseline Comparison — SVM, Random Forest, Rule-based

In [ ]:
svm_model = SVC(probability=True, random_state=RANDOM_STATE).fit(X_train_sel, y_train)
rf_model = RandomForestClassifier(random_state=RANDOM_STATE).fit(X_train_sel, y_train)

def rule_based_predict(X):
    # Simple rule: cholesterol > 240 -> at risk
    col = "chol" if "chol" in X.columns else None
    if col is None:
        return np.zeros(len(X))
    return (X[col] > (240 - scaler.mean_[continuous_cols.index("chol")]) / scaler.scale_[continuous_cols.index("chol")]).astype(int)

results = []
for name, model in [("Logistic Regression", best_lr), ("SVM", svm_model), ("Random Forest", rf_model)]:
    pred = model.predict(X_test_sel)
    prob = model.predict_proba(X_test_sel)[:, 1]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

rule_pred = rule_based_predict(X_test_sel)
results.append({
    "Model": "Rule-based (chol>240)",
    "Accuracy": accuracy_score(y_test, rule_pred),
    "F1": f1_score(y_test, rule_pred),
    "ROC-AUC": np.nan
})

results_df = pd.DataFrame(results)
print("Baseline comparison:")
results_df

## ১২. মডেল সেভ করা (লাইভ ডেমোর জন্য)
`.pkl` ফাইলে model, scaler, selected features, ও continuous_cols — সবকিছু সেভ হবে,
যেটা পরে Streamlit app লোড করবে।

In [ ]:
artifact = {
    "model": best_lr,
    "scaler": scaler,
    "selected_features": selected_features,
    "continuous_cols": continuous_cols,
    "all_columns": X.columns.tolist(),
    "categorical_cols": categorical_cols
}

joblib.dump(artifact, "heart_disease_model.pkl")
print("Saved: heart_disease_model.pkl")

# Download the file (Colab)
from google.colab import files
files.download("heart_disease_model.pkl")

### পরের ধাপ
1. উপরের সব সেল রান করো (Runtime → Restart session and run all) — CSV আপলোড চাইলে আবার আপলোড করো, শেষে `heart_disease_model.pkl` ডাউনলোড হবে।
2. সেই `.pkl` ফাইল Streamlit app-এর ফোল্ডারে রাখো (`app.py` এর পাশে)।
3. `streamlit run app.py` দিয়ে লোকাল অথবা Streamlit Community Cloud-এ ডিপ্লয় করে ক্লাসে লাইভ ডেমো দাও।